# Why experiment? The 2x2 mindset

**Source worksheet:** [yint.org/w1](https://yint.org/w1) - week 1 of a twelve-week applied DoE course for industry practitioners.

Most teams already run experiments. Few of them run *designed*
experiments. The promise of Design of Experiments is that a small,
structured set of runs answers several questions at once, with
quantified uncertainty, and points to where the next experiment should
be.

Module 1 starts at the foundations: how to name what you are varying,
how to read a 2x2 (four runs, two factors) directly from the data,
and how those four numbers already tell you the **main effect** of each
factor, whether they **interact**, and where to run a fifth experiment
to do better than any of the four you already have.

The two worked examples below come from the week-1 worksheet. Use the
*Check yourself* prompts before reading each solution; that is where
the learning sticks.

## Question 1 - the vocabulary of an experiment

A food company wants to find which inputs drive **mouth feel**, the
subjective score given by a tasting panel. For each line below, fill in
one of:

> **factor**, **objective**, **response**, **numerical**, **categorical**.

1. The product is prepared at low or high pressure. *This is an additional ____.*
2. We measure the property *P* on the product. *P could also be a ____.*
3. We want to maximize the average mouth-feel score. *This is the ____.*
4. The product was prepared by adding ingredient *F*, or not. *F is a ____ factor.*
5. The product was prepared with either 20 mg/L or 30 mg/L of ingredient *G*. *G is a ____ factor.*

## Question 2 - a two-factor yogurt study

Friends making yogurt at home run a 2x2 experiment to find the tastiest
batch. Two factors, each at two levels:

- **A = fat content of the starter yogurt** [0% or 2%]
- **B = fermentation time** [10 hours or 16 hours]

They rate each of the four batches on a 1-to-10 taste scale:

| A (fat %) | B (hours) | Rating |
|----------:|----------:|-------:|
|     0     |    10     |   5    |
|     2     |    10     |   8    |
|     0     |    16     |   6    |
|     2     |    16     |   9    |

The worksheet asks four things:

1. Add contour lines to the cube plot.
2. Calculate the average effect of the fat content of the starter yogurt.
3. Calculate the average effect of fermentation time.
4. Where should the next batch be run to push the taste score to 10?

We will answer all four with code.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

from process_improve.experiments import c, gather, lm, main_effects_plot, predict
from process_improve.experiments.visualization import visualize_doe

pio.renderers.default = "notebook_connected"

# Build the design in coded units: -1 is the low level, +1 the high level.

A = c(-1, +1, -1, +1, lo=0, hi=2, name="A", units="% fat")
B = c(-1, -1, +1, +1, lo=10, hi=16, name="B", units="hours")
y = c(5, 8, 6, 9, name="y")
yogurt = gather(A=A, B=B, y=y)
yogurt

In [ ]:
# The square_plot is the 2-factor analogue of a cube plot: response
# values at the four corners of the design.

square = visualize_doe(
    plot_type="square_plot",
    design_data=yogurt.to_dict(orient="records"),
    response_column="y",
    factors_to_plot=["A", "B"],
    factor_labels={"A": "Fat content [%]", "B": "Time [hrs]"},
    backend="plotly",
)
fig = go.Figure(square["plotly"])
fig.update_layout(width=520, height=440, title="Yogurt 2x2: corner taste scores")
fig

In [ ]:
# Fit a 2x2 model: intercept, two main effects, and the A:B interaction.

model = lm("y ~ A + B + A:B", yogurt)
params = model.get_parameters(drop_intercept=False)
print(params.to_string())

In [ ]:
# A main effect in coded units is twice the corresponding coefficient,
# because the model is centered and the factor moves from -1 to +1
# (a span of 2 units).

effect_A = 2 * params["A"]
effect_B = 2 * params["B"]
interaction_AB = 2 * params["A:B"]

print(f"Main effect of fat content (A) : {effect_A:+.2f} taste points")
print(f"Main effect of fermentation (B): {effect_B:+.2f} taste points")
print(f"Interaction A:B                : {interaction_AB:+.2f}")

In [ ]:
# Main-effects plot: average response at each level of each factor.

main_effects_plot(model, factor_labels={"A": "Fat content", "B": "Time"})

In [ ]:
# The model predicts higher ratings if we push both factors above
# their +1 levels. Aiming for a rating of 10 we extrapolate to
# A=+1.5 (fat = 2.5%), B=+1.5 (time = 17.5 hrs).

predicted = float(predict(model, A=1.5, B=1.5).iloc[0])
print(f"Predicted rating at A=+1.5 (2.5% fat), B=+1.5 (17.5 hrs): {predicted:.2f}")

## Question 3 - the bioreactor

A researcher running a bioreactor wants to maximize the **conversion**
of raw material to product. Unconverted raw material is wasted profit.
Each experiment in the table below is the average of two duplicated
runs.

| T (°C) | S (g/L) | Conversion [%] |
|-------:|--------:|---------------:|
|   35   |  1.75   |       60       |
|   41   |  1.25   |       64       |
|   41   |  1.75   |       69       |
|   35   |  1.25   |       53       |

The worksheet asks:

a. Draw a large cube plot of the system (use the full space; put
   *T* on the vertical axis).
b. Add contour lines.
c. Estimate the conversion at the center point *S* = 1.50 g/L,
   *T* = 38 °C.
d. Calculate the average effect of temperature.
e. Calculate the average effect of substrate concentration.
f. Where would you run the next experiment to improve conversion?

In [ ]:
# Coded units. A is *substrate* (S), B is *temperature* (T). Real-unit
# ranges and units are attached for clarity. (Putting temperature on
# the y-axis just means picking B for the second factor.)

S = c(-1, +1, +1, -1, lo=1.25, hi=1.75, name="S", units="g/L")
T = c(-1, +1, -1, +1, lo=35,   hi=41,   name="T", units="degC")
y = c(53, 69, 64, 60,                 name="y")
bio = gather(S=S, T=T, y=y)
bio

In [ ]:
square = visualize_doe(
    plot_type="square_plot",
    design_data=bio.to_dict(orient="records"),
    response_column="y",
    factors_to_plot=["S", "T"],
    factor_labels={"S": "Substrate [g/L]", "T": "Temperature [degC]"},
    backend="plotly",
)
fig = go.Figure(square["plotly"])
fig.update_layout(width=520, height=440, title="Bioreactor 2x2: conversion at each corner")
fig

In [ ]:
model = lm("y ~ S + T + S:T", bio)
params = model.get_parameters(drop_intercept=False)
print(params.to_string())

effect_S = 2 * params["S"]
effect_T = 2 * params["T"]
interaction_ST = 2 * params["S:T"]

print(f"\nMain effect of substrate concentration (S): {effect_S:+.2f} percentage points")
print(f"Main effect of temperature (T)            : {effect_T:+.2f} percentage points")
print(f"Interaction S:T                            : {interaction_ST:+.2f}")

center = float(predict(model, S=0, T=0).iloc[0])
print(f"\nPredicted conversion at the center point (S=1.50 g/L, T=38 degC): {center:.2f}%")

In [ ]:
# Next experiment: both effects are positive, with temperature the
# larger driver. Move along the gradient, e.g. half a step beyond +1
# in T and a quarter step beyond +1 in S. In real units:
#   T_next = 41 + 0.5 * (41 - 38) = 42.5 degC
#   S_next = 1.75 + 0.25 * (1.75 - 1.50) = 1.81 g/L

next_S, next_T = 1.25, 1.5
predicted = float(predict(model, S=next_S, T=next_T).iloc[0])
print(f"Predicted conversion at S=+{next_S} ({1.5 + next_S*0.25:.2f} g/L), "
      f"T=+{next_T} ({38 + next_T*3:.1f} degC): {predicted:.2f}%")

## Wrap-up

Four runs of a 2x2 design gave you:

- A clean vocabulary for the experiment (factors, levels, responses,
  objective).
- Two **main effects** per study, calculated directly from the corner
  averages.
- A check on whether the factors **interact** (yogurt: no; bioreactor:
  small effect worth tracking).
- An estimate of the **center-point** conversion without running it,
  and a direction for the **next experiment**.

**Next:** Module 2 fits the same models with coded units and a small
amount of linear algebra, which generalizes the manual calculation
into something that scales beyond two factors and copes with replicates,
noise, and confidence intervals.